# hopmil — runner Kaggle (E2: experimentos de imagem em GPU)

Roda a comparação dos 4 agregadores nos datasets de imagem, na GPU do Kaggle,
logando no W&B (projeto `artigo-3`) e salvando os CSVs em `/kaggle/working` para
download.

## Antes de rodar (uma vez)
1. **Settings → Accelerator → GPU**.
2. **Settings → Internet → On** (para clonar o repo e baixar MNIST/CIFAR).
3. **Add-ons → Secrets**: criar `WANDB_API_KEY` com sua chave do W&B.
4. **Add Input**: adicionar os datasets reais de histopatologia (colon / UCSB)
   como *Datasets* do Kaggle e ajustar `COLON_ROOT` / `UCSB_ROOT` abaixo.
5. Editar `REPO_URL` para o seu repositório GitHub.

## Fluxo
- Rode com `QUICK = True` primeiro: faz um teste minúsculo em **todos** os
  datasets, no projeto `artigo-3-smoke`, só para confirmar que loga certo.
- Depois `QUICK = False`: roda a sequência completa (10-fold) no projeto
  `artigo-3`.

In [ ]:
# --- Parâmetros (edite aqui) ---
REPO_URL = "https://github.com/waks38/artigo3.git"
BRANCH = "master"

# Raízes dos datasets reais (mounts de /kaggle/input). Se não existirem, são puladas.
COLON_ROOT = "/kaggle/input/crchistophenotypes/CRCHistoPhenotypes_2016_04_28"
# UCSB: o dataset andrewmvd monta aninhado em datasets/<autor>/<slug> e as imagens
# .tif ficam em Images/ (as .TIF maiúsculas em Masks/ são ignoradas pelo loader).
UCSB_ROOT = "/kaggle/input/datasets/andrewmvd/breast-cancer-cell-segmentation"

QUICK = True   # True = teste rápido (todos os datasets, mínimo); False = run completo

In [ ]:
# --- Clonar + instalar (com o extra hopfield) ---
import os
os.environ["PYTHONUTF8"] = "1"
# Sair da pasta antes de apagá-la: num 2o Run All o kernel pode estar DENTRO de
# /kaggle/working/hopmil, e apagar o cwd quebra o getcwd (clone/pip falham).
%cd /kaggle/working
!rm -rf /kaggle/working/hopmil
!git clone -q --branch $BRANCH $REPO_URL /kaggle/working/hopmil
%cd /kaggle/working/hopmil
!pip install -q -e ".[hopfield]"
print("commit:")
!git rev-parse --short HEAD

In [ ]:
# --- W&B via Kaggle Secret ---
from kaggle_secrets import UserSecretsClient
os.environ["WANDB_API_KEY"] = UserSecretsClient().get_secret("WANDB_API_KEY")
print("WANDB_API_KEY carregado:", bool(os.environ.get("WANDB_API_KEY")))

In [ ]:
# --- Rodar os experimentos de imagem ---
import subprocess, glob, shutil

project = "artigo-3-smoke" if QUICK else "artigo-3"
common = [
    "python", "-m", "hopmil.eval.compare",
    "trainer.accelerator=gpu", "trainer.devices=1",
    "n_jobs=1",                # GPU única -> fits sequenciais
    "cv.n_repeats=1",          # 10-fold simples para imagem
    "wandb.mode=online", f"wandb.project={project}",
]
if QUICK:
    common += ["cv.n_folds=2", "trainer.max_epochs=2", "early_stopping.patience=2"]

jobs = [
    {"data": "mnist_bags"},
    {"data": "fashion_mnist_bags"},
    {"data": "cifar_bags"},
    {"data": "colon_cancer", "root": COLON_ROOT},
    {"data": "ucsb_breast", "root": UCSB_ROOT},
]

for j in jobs:
    args = list(common) + [f"data={j['data']}"]
    if "root" in j:
        if not os.path.isdir(j["root"]):
            print(f"== PULANDO {j['data']}: root não encontrado ({j['root']}) ==")
            continue
        args.append(f"data.root={j['root']}")
    print("\n>>>", " ".join(args), flush=True)
    subprocess.run(args, check=False)

In [ ]:
# --- Coletar os CSVs em /kaggle/working/results (vira output baixável) ---
os.makedirs("/kaggle/working/results", exist_ok=True)
for f in glob.glob("/kaggle/working/hopmil/results/*.csv"):
    shutil.copy(f, "/kaggle/working/results/")
print("CSVs salvos:")
!ls -la /kaggle/working/results

## Baixar os resultados
- **W&B**: runs no projeto escolhido (um por dataset, nome = dataset).
- **CSVs**: aba *Output* do notebook → `results/` (ou *Save Version* para persistir
  como output da versão). Cada dataset gera `*_folds.csv`, `*_summary.csv`,
  `*_bayesian.csv` com a suíte completa de métricas.